# PPO, SAC, and learning from a fixed dataset

All these neural agents see only the current board. Their policy outputs four logits, one for each move. Invalid actions are masked. `agents/neural.py` contains the shared network; each algorithm's math is in its own agent file.

```mermaid
flowchart LR
 G[vector_game.py] --> P[ppo_train.py fresh rollout]
 G --> R[offline_data.py replay buffer]
 R --> S[sac_train.py]
 D[Saved complete games] --> O[offline_train.py]
 O --> A[agents/awr.py]
 O --> I[agents/iql.py]
 O --> C[agents/cql.py]
 P --> E[evaluate.py raw score]
 S --> E
 A --> E
 I --> E
 C --> E
```

A rollout batch and a gradient minibatch are different. Default PPO collects 256 games × 64 moves = 16,384 transitions. Four epochs with minibatches of 4,096 make 16 gradient updates. Increasing to 16,384 gives four updates per rollout.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'rl2048').exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import torch
from rl2048.agents.ppo import advantages, clipped_policy_loss
from rl2048.agents.sac import soft_value
from rl2048.agents.iql import expectile_loss
from rl2048.agents.cql import conservative_penalty
from rl2048.agents.neural import masked_log_probs


## PPO: reuse a rollout carefully

$\delta_t=r_t+\gamma(1-\mathrm{terminated}_t)V(s_{t+1})-V(s_t)$.

$A_t=\delta_t+\gamma\lambda(1-\mathrm{done}_t)A_{t+1}$.

The probability ratio is $\rho=\pi_{new}(a|s)/\pi_{old}(a|s)$. Maximize $\min(\rho A,\mathrm{clip}(\rho,0.8,1.2)A)$. The code returns the **negative** because the optimizer minimizes.

Timeouts bootstrap their final next state but cut the GAE recurrence before an auto-reset. Natural terminal states never bootstrap.

In [ ]:
new_logp=torch.tensor([1.5,1.5]).log()
old_logp=torch.zeros(2)
advantage=torch.tensor([2.,-2.])
print('PPO loss:',clipped_policy_loss(new_logp,old_logp,advantage).item())
# First sample: min(3,2.4)=2.4. Second: min(-3,-2.4)=-3.
# Negative average = 0.3.
rewards=np.array([[1.,1.],[2.,2.]],np.float32)
values=np.array([[3.,3.],[4.,4.]],np.float32)
next_values=np.array([[4.,4.],[99.,5.]],np.float32)
term=np.array([[False,False],[True,False]])
trunc=np.array([[False,False],[False,True]])
print('GAE:',advantages(rewards,values,next_values,term,trunc,gamma=.5,lam=1)[0])

## SAC: actor, critic, value, and buffer

Our explicit-value variant has two critics $Q_1,Q_2$, actor $\pi$, state value $V$, and slowly moving target $\bar V$.

- Critic target: $y=r+\gamma(1-\mathrm{terminated})\bar V(s')$.
- Value target: $\sum_a\pi(a|s)[\min(Q_1,Q_2)(s,a)-\alpha\log\pi(a|s)]$.
- Actor minimizes: $\sum_a\pi(a|s)[\alpha\log\pi(a|s)-\min(Q_1,Q_2)(s,a)]$.

The buffer stores the actual final next state, never the reset board. Modern discrete SAC can compute the soft value directly, but this version keeps the separate V you asked to examine.

In [ ]:
mask=torch.tensor([[True,True,False,False]])
logs=masked_log_probs(torch.zeros(1,4),mask)
q=torch.tensor([[2.,4.,999.,999.]])
print('Soft V:',soft_value(logs,q,.1).item(), '= 3 + 0.1 log(2)')

## Offline RL: no new game interactions during fitting

The fixed dataset has full games from our local expert and a random policy. All offline methods sample the same transitions.

**BC:** minimize $-\log\pi(a_D|s)$.

**AWR:** regress $V(s)$ to complete discounted return $G$. Set $A=G-V(s)$; minimize $-\min(\exp(A/\beta),20)\log\pi(a_D|s)$. This version uses Monte Carlo returns.

**IQL:** fit $V$ with expectile loss $|\tau-\mathbf{1}[u<0]|u^2$, where $u=\min(\bar Q_1,\bar Q_2)(s,a_D)-V(s)$. Fit critics toward $r+\gamma V(s')$. Extract the actor with weights $\min(\exp(\beta(Q-V)),100)$. The beta convention is inverse temperature here, unlike AWR's denominator.

**CQL:** add $\alpha_{CQL}[\log\sum_{a\;legal}\exp Q(s,a)-Q(s,a_D)]$ to the Bellman loss. This discourages assigning large values to unsupported actions. Our discrete version uses a Double-DQN target and a greedy critic policy.

An accurate imitation loss is not sufficient to guarantee long-game success: small action errors can lead to boards that the dataset rarely contains.

In [ ]:
print('Expectile loss for errors [-2,+2], tau=.7:',expectile_loss(torch.tensor([-2.,2.]),.7).item())
q=torch.tensor([[0.,999.,0.,999.]])
masks=torch.tensor([[True,False,True,False]])
print('CQL penalty:',conservative_penalty(q,torch.tensor([0]),masks).item(),'= log(2)')
# AWR: return12, value10, temperature2 -> exp(1)=2.718 weight.
print('AWR example weight:',np.exp((12-10)/2))

## What to graph

Primary measures are held-out raw score, maximum-tile distribution, tile-reaching rates, and game length. Put environment transitions on the x axis for online learning; offline learning has zero new environment transitions, so use optimizer updates or sampled training examples. Also record elapsed time.

Loss is a debugging signal. Different algorithms have different losses and reward scales; a lower number does not mean a stronger player.

Experiments change one factor: gamma .99/.999/1, minibatch 4,096/16,384, bottom-left/top-right potential shaping, constant survival reward, or symmetry augmentation. Final conclusions need multiple training seeds.

In [ ]:
import json
for path in sorted((ROOT/'runs/research').glob('*/validation.json')):
    data=json.loads(path.read_text())
    if 'summary' in data:
        print(path.parent.name,round(data['summary']['mean_score']),data['summary']['episodes'],'games')

Sources: [PPO](https://arxiv.org/abs/1707.06347), [SAC with V](https://arxiv.org/abs/1801.01290), [discrete SAC](https://arxiv.org/abs/1910.07207), [AWR](https://arxiv.org/abs/1910.00177), [IQL](https://arxiv.org/abs/2110.06169), [CQL](https://arxiv.org/abs/2006.04779).

Read the implementation in this order: `agents/neural.py` → one algorithm file → its training loop → `evaluate.py`. Change one parameter, save to a new output directory, and compare scores on the same evaluation seeds.